<a href="https://colab.research.google.com/github/AR-Ashik-9997/Universal-ML-Template/blob/main/Universal_ML_Template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, root_mean_squared_error, r2_score

# Classifier and Regressor imports
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression

In [ ]:
from math import inf
class UniversalMLPipeline:
  def __init__(self,data_path,target_column,problem_type="'classification'"):
    self.data_path=data_path
    self.target_column=target_column
    self.problem_type=problem_type.lower()
    self.data=None
    self.X_train=None,
    self.X_test=None,
    self.y_train=None,
    self.y_test = None,
    self.preprocessor = None
    self.best_model = None

# data loading function
  def load_data(self):
    print("Data Loading.....")
    self.data=pd.read_csv(self.data_path)
    print(f"Data shape:{self.data.shape}")
    # display(self.data.head())
    return self.data.head()

 # preprocessing pipeline
  def build_preprocessing_pipeline(self,X):
    num_cols=X.select_dtypes(include=["int64","float64"]).columns.tolist()
    cat_cols=X.select_dtypes(include=["object"]).columns.tolist()

    num_pipeline=Pipeline(
        steps=[
            ("imputer",SimpleImputer(strategy='median')),
            ("scaler",StandardScaler()),
        ])

    cat_pipeline=Pipeline(
        steps=[
            ("imputer",SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ])

  # ColumnTransform
    self.preprocessor=ColumnTransformer(
        transformers=[
            ("num",num_pipeline,num_cols),
            ('cat',cat_pipeline,cat_cols)
        ],
        remainder="passthrough"

        )

# data Splitting
  def split_data(self,test_size=0.2,randome_state=42):

    x=self.data.drop(columns=[self.target_column],axis=1)
    y=self.data[self.target_column]

    self.build_preprocessing_pipeline(x)

    stratify=y if self.problem_type=='classification' else None

    self.X_train,self.X_test,self.y_train,self.y_test=train_test_split(x,y,test_size=test_size,random_state=randome_state,stratify=stratify)

    self.X_train=self.preprocessor.fit_transform(self.X_train)
    self.X_test=self.preprocessor.transform(self.X_test)
    print("Completed Preprocessing")


# data Splitting
  def model_train_evaluting(self):

    if self.problem_type=='classification':
      models={
          "LogisticRegression":LogisticRegression(max_iter=1000),
          "RandomForestClassifier":RandomForestClassifier(random_state=42)
      }

      best_score = 0.0

    elif self.problem_type=='regression':
      models={
          'LinearRegression':LinearRegression(),
          'RandomForestRegressor':RandomForestRegressor(random_state=42)
      }
      best_score = float('inf')
    else:
      raise ValueError("Only Classification and regression algorithm is allowed ")

    self.best_model=None

    for name,model in models.items():
      model.fit(self.X_train,self.y_train)
      prediction=model.predict(self.X_test)

      if self.problem_type=='classification':
        acc=accuracy_score(self.y_test,prediction)
        print(f"{name} Accuracy: {acc:.4f}")

        if acc>best_score:
          best_score=acc
          self.best_model=model

        else:
          rmse=root_mean_squared_error(self.y_test,prediction)
          print(f"{name} RMSE: {rmse:.4f}")

          if rmse<best_score:
            best_score=rmse
            self.best_model=model

    print(f"Best model is {self.best_model.__class__.__name__}")


In [ ]:
if __name__=="__main__":
  DATA_PATH="/content/heart_disease.csv"
  TARGET_COLUMN="Disease"
  PROBLEM="Classification"
pipeline=UniversalMLPipeline(data_path=DATA_PATH,target_column=TARGET_COLUMN,problem_type=PROBLEM)
try:
  pipeline.load_data()
  pipeline.split_data()
  pipeline.model_train_evaluting()
except FileNotFoundError:
  print("File note found")

Data Loading.....
Data shape:(270, 14)
Completed Preprocessing
LogisticRegression Accuracy: 0.8519
RandomForestClassifier Accuracy: 0.8148
RandomForestClassifier RMSE: 0.4303
Best model is:RandomForestClassifier
